# Synthetic CRISPR Screen Hit Calling for NBX-042 Resistance

As the bioinformatics scientist on project `SYN-NBX042-CRISPR-001`, I use this notebook to demonstrate a deterministic hit-calling workflow for a pooled CRISPR knockout screen under drug selection. All guide-count data below are synthetically generated in the notebook; no external input files are required.

In [2]:
import math
import random
from statistics import median

RANDOM_SEED = 42042
rng = random.Random(RANDOM_SEED)

PROJECT_ID = "SYN-NBX042-CRISPR-001"
COMPOUND = "NBX-042"
N_GENES = 250
GUIDES_PER_GENE = 4
BASELINE_SAMPLES = ["NBX042_BASE_R1", "NBX042_BASE_R2"]
TREATED_SAMPLES = ["NBX042_DRUG_R1", "NBX042_DRUG_R2"]

HIT_EFFECT_LOG2 = {
    "TP53": 1.10,
    "NF1": 1.24,
    "KEAP1": 1.58,
    "SAFE_AAVS1": 0.12,
}

print(f"project_id {PROJECT_ID}")
print(f"compound {COMPOUND}")
print(f"genes {N_GENES}")
print(f"guides_per_gene {GUIDES_PER_GENE}")
print("primary_resistance_hit KEAP1")

project_id SYN-NBX042-CRISPR-001
compound NBX-042
genes 250
guides_per_gene 4
primary_resistance_hit KEAP1


## Screen Design

I modeled a compact genome-scale pilot with 250 genes and 4 sgRNAs per gene. Two baseline replicates represent the library before selection, and two `NBX-042` drug-treated replicates represent cells that survived compound pressure. The intended resistance markers are `TP53`, `NF1`, and `KEAP1`, with `KEAP1` simulated as the strongest resistance marker.

`SAFE_AAVS1` is included as a safe-targeting control because perturbing that locus is expected to be tolerated. In a real screen, these controls help me separate biological enrichment from library bottlenecks, neutral guide behavior, and sample normalization artifacts.

In [3]:
priority_genes = list(HIT_EFFECT_LOG2)
background_genes = [
    f"GENE_{i:04d}" for i in range(1, N_GENES - len(priority_genes) + 1)
]
genes = priority_genes + background_genes

def count_from_expected(expected):
    noisy = expected * rng.lognormvariate(0.0, 0.055)
    return max(1, int(round(noisy)))

counts = []
for gene in genes:
    gene_bias = rng.gauss(0.0, 0.05)
    gene_effect = HIT_EFFECT_LOG2.get(gene, rng.gauss(0.0, 0.16)) + gene_bias
    for guide_number in range(1, GUIDES_PER_GENE + 1):
        baseline_mean = rng.lognormvariate(7.85, 0.35)
        guide_effect = gene_effect + rng.gauss(0.0, 0.08)
        row = {"gene": gene, "guide_id": f"{gene}_sg{guide_number:02d}"}
        for sample in BASELINE_SAMPLES:
            row[sample] = count_from_expected(baseline_mean)
        for sample in TREATED_SAMPLES:
            row[sample] = count_from_expected(baseline_mean * (2 ** guide_effect))
        counts.append(row)

print(f"{'gene':<10} {'guide_id':<16} {'BASE_R1':>7} {'BASE_R2':>7} {'DRUG_R1':>7} {'DRUG_R2':>7}")
for row in counts[:8]:
    print(
        f"{row['gene']:<10} {row['guide_id']:<16} "
        f"{row['NBX042_BASE_R1']:>7} {row['NBX042_BASE_R2']:>7} "
        f"{row['NBX042_DRUG_R1']:>7} {row['NBX042_DRUG_R2']:>7}"
    )

gene       guide_id         BASE_R1 BASE_R2 DRUG_R1 DRUG_R2
TP53       TP53_sg01           4282    4775    9894   10741
TP53       TP53_sg02           3767    3642    7993    8803
TP53       TP53_sg03           2211    2249    4724    4920
TP53       TP53_sg04           3289    2934    6528    7656
NF1        NF1_sg01            1636    1861    4412    4084
NF1        NF1_sg02            2910    2763    6011    6399
NF1        NF1_sg03            1555    1659    3756    3580
NF1        NF1_sg04            2948    2889    8515    7833


## Guide-Level Enrichment

The central guide-level statistic is a pseudocount-stabilized log2 ratio of treated abundance over baseline abundance. I keep the function name explicit so that the computation is easy to retrieve and audit: `compute_guide_enrichment`.

In [4]:
def compute_guide_enrichment(count_table, baseline_cols, treatment_cols, pseudocount=1.0):
    enriched = []
    for row in count_table:
        baseline_mean = sum(row[col] for col in baseline_cols) / len(baseline_cols)
        treated_mean = sum(row[col] for col in treatment_cols) / len(treatment_cols)
        out = dict(row)
        out["baseline_mean"] = baseline_mean
        out["treated_mean"] = treated_mean
        out["log2_enrichment"] = math.log2(
            (treated_mean + pseudocount) / (baseline_mean + pseudocount)
        )
        enriched.append(out)
    return enriched

guide_enrichment = compute_guide_enrichment(counts, BASELINE_SAMPLES, TREATED_SAMPLES)

print(f"{'gene':<10} {'guide_id':<16} {'baseline_mean':>13} {'treated_mean':>12} {'log2_enrichment':>15}")
for row in guide_enrichment[:6]:
    print(
        f"{row['gene']:<10} {row['guide_id']:<16} "
        f"{row['baseline_mean']:>13.1f} {row['treated_mean']:>12.1f} "
        f"{row['log2_enrichment']:>15.3f}"
    )

gene       guide_id         baseline_mean treated_mean log2_enrichment
TP53       TP53_sg01               4528.5      10317.5           1.188
TP53       TP53_sg02               3704.5       8398.0           1.181
TP53       TP53_sg03               2230.0       4822.0           1.112
TP53       TP53_sg04               3111.5       7092.0           1.188
NF1        NF1_sg01                1748.5       4248.0           1.280
NF1        NF1_sg02                2836.5       6205.0           1.129


In [5]:
def mean(values):
    return sum(values) / len(values)

def bh_qvalues(p_values):
    indexed = sorted(enumerate(p_values), key=lambda item: item[1])
    n = len(indexed)
    adjusted = [0.0] * n
    running = 1.0
    for rank_from_end, (idx, p_value) in enumerate(reversed(indexed), start=1):
        rank = n - rank_from_end + 1
        running = min(running, p_value * n / rank)
        adjusted[idx] = min(1.0, running)
    return adjusted

by_gene = {}
for row in guide_enrichment:
    by_gene.setdefault(row["gene"], []).append(row)

gene_scores = []
for gene, rows in by_gene.items():
    enrichments = [row["log2_enrichment"] for row in rows]
    gene_scores.append(
        {
            "gene": gene,
            "mean_log2_enrichment": mean(enrichments),
            "median_log2_enrichment": median(enrichments),
            "guide_count": len(rows),
            "baseline_mean": mean([row["baseline_mean"] for row in rows]),
            "treated_mean": mean([row["treated_mean"] for row in rows]),
        }
    )

null_values = [
    row["mean_log2_enrichment"]
    for row in gene_scores
    if row["gene"] not in priority_genes
]
null_center = median(null_values)
null_mad = median([abs(value - null_center) for value in null_values])
null_sigma = 1.4826 * null_mad or 0.01

for row in gene_scores:
    z_score = (row["mean_log2_enrichment"] - null_center) / null_sigma
    row["p_value"] = 0.5 * math.erfc(z_score / math.sqrt(2))

q_values = bh_qvalues([row["p_value"] for row in gene_scores])
for row, q_value in zip(gene_scores, q_values):
    row["q_value"] = q_value
    row["resistance_hit"] = (
        row["q_value"] < 0.05 and row["mean_log2_enrichment"] > 0.50
    )

gene_scores.sort(
    key=lambda row: (row["resistance_hit"], row["mean_log2_enrichment"]),
    reverse=True,
)

selected_genes = {"KEAP1", "NF1", "TP53", "SAFE_AAVS1"}
hit_table = [
    row for row in gene_scores if row["gene"] in selected_genes or row["resistance_hit"]
]
hit_table.sort(key=lambda row: row["mean_log2_enrichment"], reverse=True)

print(f"{'gene':<12} {'mean_log2':>9} {'median_log2':>11} {'guides':>6} {'q_value':>9} {'hit':>5}")
for row in hit_table:
    print(
        f"{row['gene']:<12} {row['mean_log2_enrichment']:>9.3f} "
        f"{row['median_log2_enrichment']:>11.3f} {row['guide_count']:>6} "
        f"{row['q_value']:>9.4f} {str(row['resistance_hit']):>5}"
    )
print()
print(f"genes_passing_fdr_like_threshold {sum(row['resistance_hit'] for row in gene_scores)}")

gene         mean_log2 median_log2 guides   q_value   hit
KEAP1            1.625       1.600      4    0.0000  True
NF1              1.271       1.235      4    0.0000  True
TP53             1.167       1.184      4    0.0000  True
SAFE_AAVS1       0.192       0.144      4    0.9361 False

genes_passing_fdr_like_threshold 3


In [6]:
def correlation(x_values, y_values):
    x_mean = mean(x_values)
    y_mean = mean(y_values)
    numerator = sum(
        (x - x_mean) * (y - y_mean) for x, y in zip(x_values, y_values)
    )
    x_denominator = math.sqrt(sum((x - x_mean) ** 2 for x in x_values))
    y_denominator = math.sqrt(sum((y - y_mean) ** 2 for y in y_values))
    return numerator / (x_denominator * y_denominator)

baseline_corr = correlation(
    [row["NBX042_BASE_R1"] for row in counts],
    [row["NBX042_BASE_R2"] for row in counts],
)
drug_corr = correlation(
    [row["NBX042_DRUG_R1"] for row in counts],
    [row["NBX042_DRUG_R2"] for row in counts],
)

print(f"qc_metric baseline_replicate_correlation {baseline_corr:.3f}")
print(f"qc_metric drug_replicate_correlation {drug_corr:.3f}")
print("qc_status replicate_structure_pass")

qc_metric baseline_replicate_correlation 0.976
qc_metric drug_replicate_correlation 0.980
qc_status replicate_structure_pass


## Interpretation

The synthetic screen recovers the intended NBX-042 resistance program: `KEAP1` is the primary resistance hit, followed by `NF1` and `TP53`. Each called hit is supported by all 4 guides and passes the FDR-like rule of `q_value < 0.05` with mean log2 enrichment above 0.50.

`SAFE_AAVS1` remains below the hit threshold, which is the expected behavior for this safe-targeting control. I would use that result as a quick check that neutral control guides are not being systematically misclassified as resistance drivers.

In [7]:
primary_hit = gene_scores[0]
called_markers = [
    row["gene"]
    for row in gene_scores
    if row["resistance_hit"] and row["gene"] in {"KEAP1", "NF1", "TP53"}
]

print(f"conclusion primary_resistance_hit {primary_hit['gene']}")
print(f"conclusion validated_resistance_markers {','.join(called_markers)}")
print("conclusion safe_targeting_control SAFE_AAVS1_not_called")

conclusion primary_resistance_hit KEAP1
conclusion validated_resistance_markers KEAP1,NF1,TP53
conclusion safe_targeting_control SAFE_AAVS1_not_called
